# L2P for IP102 (Learning to Prompt, CVPR2022)
Incremental learning + retrieval (R@1/5/10, mAP) + open-world (AUROC, FPR95) + lifelong (plasticity/forgetting/overall) metrics on the IP102 pest dataset.
Dataset: **1 Input duy nhat** (chua `train.json`/`val.json`/`test.json` + thu muc anh `JPEGImages`). Code duoc clone tu GitHub qua env `IP102_CODE_REPO`, neu khong co thi tim trong `/kaggle/input`.

In [1]:
# ==== 1. Lay code tu GitHub (env IP102_CODE_REPO) hoac /kaggle/input ====
import os, sys, glob, subprocess

CODE_DIR = None
import os, sys, glob, subprocess
os.environ['IP102_CODE_REPO'] = "https://github.com/nta2112/L2P-IP102-custom"

if os.environ.get('IP102_CODE_REPO'):
    repo = os.environ['IP102_CODE_REPO']
    target = '/kaggle/working/L2P-for-IP102'
    if not os.path.isdir(target):
        print('git clone', repo)
        subprocess.run(['git', 'clone', repo, target], check=True)
    CODE_DIR = target
else:
    for base in ('/kaggle/input', '/kaggle/working'):
        found = sorted(glob.glob(os.path.join(base, '**', 'main_ip102.py'),
                                 recursive=True))
        if found:
            CODE_DIR = os.path.dirname(found[0])
            break

if not CODE_DIR:
    raise RuntimeError('Khong tim thay code. Dat env IP102_CODE_REPO '
                       'hoac day code vao /kaggle/input.')

sys.path.insert(0, CODE_DIR)
print('CODE_DIR =', CODE_DIR)

git clone https://github.com/nta2112/L2P-IP102-custom


Cloning into '/kaggle/working/L2P-for-IP102'...


CODE_DIR = /kaggle/working/L2P-for-IP102


In [2]:
# ==== 2. Cai dat thu vien JAX/Flax ====
import subprocess, sys
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

def need(mod):
    try:
        __import__(mod)
        return False
    except ImportError:
        return True

missing = [m for m in ('flax', 'jax', 'clu', 'ml_collections', 'tensorflow',
                       'scipy') if need(m)]
if missing:
    print('installing', missing)
    if 'jax' in missing and sys.platform.startswith('linux'):
        # Kaggle GPU notebooks: install JAX with CUDA 12 support
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            'jax[cuda12]', 'flax', 'clu', 'ml_collections',
                            'scipy'], check=True)
            missing = [m for m in ('flax', 'clu', 'ml_collections', 'scipy')
                       if need(m)]
        except subprocess.CalledProcessError:
            pass
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] +
                       missing, check=True)
else:
    print('all deps present')

import jax
print('jax', jax.__version__, '| devices', jax.devices())

installing ['clu']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 399.0 kB/s eta 0:00:00
jax 0.7.2 | devices [CudaDevice(id=0)]


In [3]:
# ==== 3. Kiem tra dataset (auto tim bang deep-walk) ====
from libml.ip102_data import get_data_manager

dm = get_data_manager('ip102', seed=1993, split_val=True)
report = dm.verify()
print('DATA_ROOT =', dm.data_root)
for s in ('train', 'val', 'test'):
    if s not in report:
        continue
    r = report[s]
    print('%-5s images=%-5d anns=%-5d labeled=%-5d missing=%d'
          % (s, r['images_in_json'], r['annotations'],
             r['images_labeled'], r['missing_images']))
assert report['train']['missing_images'] == 0
assert report['num_classes'] == 25
print('num_classes =', report['num_classes'], '| val_split =', report['val_split'])

DATA_ROOT = /kaggle/input/datasets/nta212/ip102-for-object-detection
train images=8664  anns=9545  labeled=8664  missing=0
val   images=2176  anns=2374  labeled=2176  missing=0
test  images=2713  anns=2946  labeled=2713  missing=0
num_classes = 25 | val_split = val


In [4]:
# ==== 4. Test nhanh: max_tasks=1 (1 task dau), 1 epoch ====
from main_ip102 import run_train

# quick = run_train(model='L2P', max_tasks=1, memory_size=0, num_epochs=1)
# print('quick results ->', quick)

In [5]:
# ==== 5. Chay du: max_tasks=0 (toan bo task) ====
# Đảm bảo model đã được định nghĩa trước khi dùng                                                        
# ==== 5. Chay du: max_tasks=0 (toan bo task) ====
# Đảm bảo model đã được định nghĩa trước khi dùng                                                        
if 'model' not in locals() and 'model' not in globals():                                                 
    from main_ip102 import run_train  # import để model có sẵn qua side effect
if 'model' in locals() or 'model' in globals():
    model.mAP_matrix = []
    print("Đã reset mAP_matrix")
else:
    print("⚠️ Chưa có biến model, đang chờ chạy cell tạo model...")
full = run_train(model='L2P', max_tasks=0, memory_size=0)
print('final results ->', full)

⚠️ Chưa có biến model, đang chờ chạy cell tạo model...
✅ Đã khởi tạo model.mAP_matrix = []
final results -> /kaggle/working/output_ip102/results.csv


## Ket qua (results.csv)
Header: `task,numclass,cnn_top1,nme_top1,R@1,R@5,R@10,mAP,AUROC,FPR95,Plasticity,Forgetting,Overall`
(`AUROC/FPR95 = NA` khi da thay du toan bo lop -> khong con OOD de do).

In [6]:
# ==== 6. Hien thi results.csv (glob dung duong dan noi code chay) ====
import glob
import pandas as pd

cands = set()
for base in ('/kaggle/working', CODE_DIR, '.'):
    cands |= set(glob.glob(os.path.join(base, '**', 'results.csv'),
                           recursive=True))
cands = sorted(cands, key=lambda p: os.path.getmtime(p))
print('results.csv files:', cands)
assert cands, 'Khong tim thay results.csv'
path = cands[-1]
print('displaying ->', path)
df = pd.read_csv(path)
display(df)

results.csv files: ['./output_ip102/results.csv', '/kaggle/working/output_ip102/results.csv']
displaying -> /kaggle/working/output_ip102/results.csv


,task,numclass,cnn_top1,nme_top1,R@1,R@5,R@10,mAP,AUROC,FPR95,Recall@1_seen,Recall@1_unseen,Plasticity,Forgetting,Overall
0,1,7,0.771242,0.630719,0.771242,0.998366,1.000000,0.712108,0.858243,0.604575,0.771242,0.0,0.712108,0.000000,0.712108
1,2,13,0.504940,0.562020,0.504940,0.987925,1.000000,0.597286,0.821912,0.540066,0.504940,0.0,0.356976,0.795785,0.356976
2,3,19,0.639070,0.646333,0.639070,0.970225,0.994916,0.670669,0.758863,0.815541,0.639070,0.0,0.240126,0.863533,0.240126
3,4,25,0.662224,0.641544,0.662224,0.962776,0.994945,0.620374,NaN,NaN,NaN,NaN,0.180948,0.867115,0.180948
